In [1]:
!pip install -q langchain langchain-groq

In [2]:
import os

from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.3
)

In [4]:
products = [
    {
        "id": 101,
        "name": "Cotton Casual T-Shirt",
        "category": "T-Shirt",
        "color": "Black",
        "price": 599,
        "sizes": ["S", "M", "L", "XL"],
        "stock": 20
    },
    {
        "id": 102,
        "name": "Slim Fit Jeans",
        "category": "Jeans",
        "color": "Blue",
        "price": 1299,
        "sizes": ["M", "L", "XL"],
        "stock": 10
    },
    {
        "id": 103,
        "name": "Hooded Sweatshirt",
        "category": "Hoodie",
        "color": "Grey",
        "price": 999,
        "sizes": ["S", "M", "L"],
        "stock": 0
    }
]

print("Product catalogue loaded successfully!")

Product catalogue loaded successfully!


In [6]:

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
import json

# Convert product catalogue into JSON
product_data = json.dumps(products, indent=2)

# Escape curly braces so LangChain treats JSON as text
product_data = product_data.replace("{", "{{").replace("}", "}}")

SYSTEM = f"""
ROLE:
You are the friendly AI shopping assistant for EcomAgent AI.

FACTS:
Here is the product catalogue:
{product_data}

RULES:
1. Answer using the available product information.
2. Never invent product names or prices.
3. Never promise product availability without checking stock.
4. Mention available sizes only when provided in the catalogue.
5. If the answer isn't in the product information,
   say "I'll check with the store owner."
6. Remember the customer's name and previous conversation.
7. Be accurate when calculating prices.

FORMAT:
Under 50 words, friendly and simple English.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

history = []

def ask(question: str) -> str:
    answer = chain.invoke({
        "history": history,
        "question": question
    })

    history.extend([
        ("human", question),
        ("ai", answer)
    ])

    return answer

print(ask("Hi, I'm Ravi. Do you have cotton casual T-shirts?"))
print(ask("What is my name?"))


Hi Ravi! Yes, we have the Cotton Casual T‑Shirt in Black. It’s ₹599, available in S, M, L, XL, and we have 20 in stock. Let me know which size you’d like!
Your name is Ravi.


In [7]:
history

[('human', "Hi, I'm Ravi. Do you have cotton casual T-shirts?"),
 ('ai',
  'Hi Ravi! Yes, we have the Cotton Casual T‑Shirt in Black. It’s ₹599, available in S, M, L, XL, and we have 20 in stock. Let me know which size you’d like!'),
 ('human', 'What is my name?'),
 ('ai', 'Your name is Ravi.')]

In [8]:
import csv

test_data = [
    {
        "question": "What is the price of Cotton Casual T-Shirt?",
        "must_include": "599"
    },
    {
        "question": "Which sizes are available in Slim Fit Jeans?",
        "must_include": "M"
    },
    {
        "question": "Is the Hooded Sweatshirt available?",
        "must_include": "out of stock"
    }
]

with open("/content/test.csv", "w", newline="") as file:

    writer = csv.DictWriter(
        file,
        fieldnames=["question", "must_include"]
    )

    writer.writeheader()
    writer.writerows(test_data)

In [9]:
def run_tests(path: str = "/content/test.csv") -> None:

    with open(path, newline="") as file:
        rows = list(csv.DictReader(file))

    passed = 0

    for row in rows:

        answer = chain.invoke({
            "history": [],
            "question": row["question"]
        })

        ok = row["must_include"].lower() in answer.lower()

        passed += ok

        print(
            "PASS" if ok else "FAIL",
            "|",
            row["question"],
            "→",
            answer[:70]
        )

    print(
        f"\n{passed}/{len(rows)} passed "
        f"({passed / len(rows):.0%})"
    )

In [10]:
run_tests()

PASS | What is the price of Cotton Casual T-Shirt? → The Cotton Casual T‑Shirt costs ₹599.
PASS | Which sizes are available in Slim Fit Jeans? → Sure! The Slim Fit Jeans are available in sizes **M, L, and XL**.
PASS | Is the Hooded Sweatshirt available? → The Hooded Sweatshirt is currently out of stock. Let me know if you’d 

3/3 passed (100%)


In [11]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Do you have cotton casual T-shirts?"
        }
    ]
})

print(result["messages"][-1].content)

Yes! We have the Cotton Casual T‑Shirt in Black. It’s ₹599, in sizes S, M, L, XL, and we have 20 in stock. Let me know if you’d like to add one to your cart!
